## Imports and modules

In [ ]:
import os
from pathlib import Path
import virtualizarr as vz
from virtualizarr.parsers import KerchunkParquetParser
import warnings

# Automatically tells OS to think that this notebook is at repo root. 
# No need to further prefix relative paths
repo_root = Path(Path.cwd()).parent
os.chdir(repo_root)

from utils.config_utils import (
    check_runtime_readiness,
    load_pipeline_config,
    resolve_secrets,
)
from pipeline.inventory import build_inventory_snapshot_and_diff
from pipeline.generate_parquet import _build_registry, reference_relpath_for_key
from pipeline.ecmwf_consolidate import (
    FlowInventory,
    StagingConfig,
    source_key_sorting,
    unusable_ecmwf_reference_keys,
    _probe_dataset_using_parquet,
)

print("Imports OK")

In [ ]:
check_runtime_readiness()
kp = load_pipeline_config("configs/config.yaml")
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp)

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## S3 objects inventory building

In [ ]:
ledger = build_inventory_snapshot_and_diff(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
)

print("Inventory summary:", ledger["summary"])

inventory_diff = ledger["diff"]
inventory_objects = ledger["current_objects"]
pending_ledger = ledger["next_ledger"]
previous_objects =  ledger["previous_ledger"].get("objects", {})

## vz.open_mfdataset() uses file URI
Cannot intake Path() or str(ref_path)

This is now fixed with `path.resolve().as_uri()`

In [ ]:
registry = _build_registry(kp, ACCESS_KEY, SECRET_KEY)
staging = StagingConfig(
    staging_volume_path=Path(kp["output"]["staging_volume_path"])
)

ecmwf_inventory = FlowInventory(
    current_objects=inventory_objects,
    previous_objects=previous_objects,
    flow_id="ecmwf_weekly_nc",
)

ecmwf_keys = source_key_sorting(
    ecmwf_inventory.current_objects,
    ecmwf_inventory.flow_id,
)

probe_failures = []

for source_key in ecmwf_keys:
    ref_path = staging.staging_volume_path / reference_relpath_for_key(source_key)
    try:
        _probe_dataset_using_parquet(ref_path, registry=registry)
    except Exception as exc:
        probe_failures.append(
            {
                "source_key": source_key,
                "ref_path": str(ref_path),
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
        )

if probe_failures:
    print("Probe failures:")
    for failure in probe_failures[:10]:
        print(failure)
    raise RuntimeError(
        f"_probe_dataset_using_parquet failed for {len(probe_failures)} ECMWF refs"
    )

unusable = unusable_ecmwf_reference_keys(
    inventory=ecmwf_inventory,
    staging=staging,
    registry=registry,
)

if unusable:
    print("Unusable ECMWF keys:")
    for key in unusable:
        print(key)
    raise RuntimeError(
        f"unusable_ecmwf_reference_keys returned {len(unusable)} unusable refs"
    )

print(
    {
        "status": "ok",
        "flow_id": ecmwf_inventory.flow_id,
        "probed_refs": len(ecmwf_keys),
        "unusable_keys": len(unusable),
        "staging": str(staging.staging_volume_path),
    }
)

## Inconsitent chunking - cannot mfdataset()
The below cell will try to `vz.open_virtual_mfdataset()` on the first 4 weekly files then `vds.to_kerchunk()`

This fails with:
> ValueError: Cannot concatenate arrays with partial chunks because only regular chunk grids are currently supported. Concat input 0 has array length 14 along the concatenation axis which is not evenly divisible by chunk length 4

### Bad idea - Do not do: Force VirtualiZarr to treat each file as single chunk
If I rechunk the time axis on the vds, using `vds = vds.chunk(time=-1)`. 

Downstream web app will load in weekly ecmwf NetCDFs as contigious chunks. I/O overhead is tremendous and performance drops significantly, negating any chunking gains

### Alternative 1:
Use a single catalog artifact (inventory.json)  for discovery and routing of weekly .parq files

### Alternative 2:
Map/Reduce with `MultiZarrToZarr`. This is a Kerchunk native implementation and should allow for inconsistent chunk grids. Although the [code to implement is somewhat complex](https://www.google.com/search?client=firefox-b-d&hs=4byp&sca_esv=8945512b0b3ca80a&sxsrf=ANbL-n6dIjEawGV7vM1JYgoeTkq_vVat8g%3A1780028825495&ntc=1&sa=X&ved=0CAIQ2_wOahcKEwiA8fGy1N2UAxUAAAAAHQAAAAAQEA&biw=878&bih=832&dpr=1.09&mtid=TwcZaqPJAcGNnesPr7SYsAU&mstk=AUtExfAZVKe4b8zUzTuAC94cLtdlraBBJTTE02MMkt1tAlQai5qcrvZXh1CUsWndtbrxsD34zlt5rX92bnBaDBVX2oyUXrb_qq3dI9QeR3E-JHmyaPbLPO-2wBWOK8cWB5-_bLt_pBR-Tx03SbNIbEqADwTW_y_YtpaKXBSufJ4vJlqIkGvRcOKM_9MBDqQCpxRAnM5A6ukAYqoAwqwfLNwfv22dvuSklF681yA_XvTzHgrQgMxeE2p0qDIaXRiXFUapaXsxHEzHNuwqH4IIW2zsbSrxCDTe6Qnr4yZ-4FBrfbcmj_v6LGtUc1EJ3P_mMOcyD-hKMTpucxbKdd02Lbu-Kvn4Gnka_0ASQxNOWNoa9rh3Af5J0YUrH5XWBGsnfTPH3E2Zq4PpKDPogrol3bvoTb9Ms2-pvWrobEY8NS-H1Cal7vjk1Aqxhk7pklvxHTRoJvKiNwOxCxk&csuir=1&aep=26&q=virtualizarr+TypeError+could+not+find+chunk+manager+which+recognises+type+%3Cclass+virtualizarr.manifests.array.Manifests+Array&atvm=2&udm=50&lns_mode=cvst)


In [ ]:
n = 4  # try first 4 weekly refs
subset_keys = ecmwf_keys[:n]

input_paths = [
    staging.staging_volume_path / reference_relpath_for_key(key)
    for key in subset_keys
]
input_uris = [path.resolve().as_uri() for path in input_paths]

output_path = Path(".tmp/ECMWF_consolidate/ecmwf_consolidated_first_4.nc.parq")
output_path.parent.mkdir(parents=True, exist_ok=True)

try:
    vds = vz.open_virtual_mfdataset(
        input_uris,
        registry=registry,
        parser=KerchunkParquetParser(),
        combine="nested",
        concat_dim="time",
        compat="override",
        coords="minimal",
        data_vars="minimal",
        parallel="dask",
        loadable_variables=[],
    )

    vds.vz.to_kerchunk(
        filepath=str(output_path),
        format="parquet",
        record_size=kp["execution"]["parquet_record_size"],
        categorical_threshold=kp["execution"]["categorical_threshold"],
    )
except ValueError as exc:
    expected = (
        "Cannot concatenate arrays with partial chunks because only regular chunk grids "
        "are currently supported."
    )
    assert expected in str(exc)
else:
    raise AssertionError("Expected ValueError for partial chunks, but none was raised.")

print(
    {
        "subset_size": n,
        "subset_keys": subset_keys,
        "output_path": str(output_path),
    }
)